# Weather Hazard Category · Model selection, final fit and promotion

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Pipeline stages 3-8 · shortlist, select features, choose, test once, deploy*

This notebook takes the tuned WHC experiments through the rest of the pipeline: it shortlists the best algorithms, selects feature families for each, chooses one champion, refits it on all development data, scores it **once** on the test split and promotes it to `models/weather_hazard/production/` for the FastAPI service. It is the **only notebook that reads the test split**.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

| Stage | What happens | Features | Data | Section |
|:-:|:--|:--|:--|:-:|
| 1–2 | Baselines and hyperparameter tuning (done in the experiment notebooks) | All 105 | Walk-forward CV | 1 |
| 3 | Shortlist the best tuned algorithms | All | – | 2 |
| 4 | Feature selection with the settings fixed, then a short re-tune | Per model | Walk-forward CV | 3 |
| 5 | Select the champion | Champion's own | CV scores | 4 |
| 6 | Final fit | Champion's own | All development data (2016–2024) | 5 |
| 7 | Test once | Champion's own | Test (2025) | 6 |
| 8 | Deploy, with a drift reference for monitoring | Champion's own | Production | 7 |

**Two searches, not one.** Each experiment notebook tuned its algorithm's settings on all
features, so every algorithm competes after tuning. Feature selection is spent only on the
shortlist, with the settings fixed, followed by a short re-tune because good settings depend on
the features.

**Why the scores are comparable.** Every experiment used the same four walk-forward folds
(validating on 2021, 2022, 2023 and 2024), so the saved cross-validated scores are compared
directly, without refitting anything.

**Why the test split is still honest.** Every decision (shortlist, features, settings, champion)
is made with cross-validation before the test split is read. It is scored once, in section 6.

> **Runtime.** Feature selection fits `(FEATURE_TRIALS + RETUNE_TRIALS) x folds x targets` models
> per shortlisted algorithm. Lower the trial counts for a quicker run.

---
## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

TASK = "weather_hazard"
SHORTLIST_SIZE = 3    # stage 3: algorithms kept for feature selection
FEATURE_TRIALS = 20   # stage 4: Optuna trials over the feature-family switches
RETUNE_TRIALS = 10    # stage 4: short re-tune of the settings on the chosen features
VERBOSE = 1           # 0 = results only, 1 = progress + every Optuna trial, 2 = every model fit

task = wx.load_task(TASK)
print(f"{task.target_name} | metric: {task.metric} | targets: {task.target_columns}")

---
## 1. Tuned experiments (stages 1-2)

Each experiment notebook saved its tuned model, its cross-validated scores and its specification.
Bundles built on an older feature set are excluded; rerun their notebooks to include them. The
ranking reads the saved scores, including the baselines' scores on the same folds.

In [ ]:
bundles = wx.discover_bundles(task)
contract = wx.check_feature_contract(bundles, task)
usable = contract["matches_current"] & contract["has_models"]
COMPARED = contract.index[usable].tolist()
if not COMPARED:
    raise RuntimeError("No bundle matches the current feature set. Rerun the experiment notebooks.")
print(f"Excluded (stale or empty): {contract.index[~usable].tolist() or 'none'}")
display(wx.bundle_inventory(bundles))

ranking = wx.rank_tuned(task, {name: bundles[name] for name in COMPARED})
wx.plot_ranking(task, ranking, f"{task.name}-tuned-ranking")
plt.show()
ranking

---
## 2. Shortlist (stage 3)

The best tuned algorithms go forward. Baselines and null models (the dummies) are listed but never
shortlisted: they are the floor, not candidates.

In [ ]:
SHORTLIST, shortlist_table = wx.shortlist(ranking, bundles, size=SHORTLIST_SIZE)
print(f"Shortlisted for feature selection: {SHORTLIST}")
shortlist_table

---
## 3. Feature selection (stage 4)

Each shortlisted experiment is restored with its tuned settings. Optuna then switches each feature
family on or off (today's persistence column is always kept) with the settings fixed, scoring
every combination with walk-forward CV. The settings are then re-tuned briefly on the chosen
features and kept only if they improve the score. Each horizon is handled separately.

The table for each experiment lists the mean and spread across folds, the features kept, the
families dropped and whether the re-tune helped.

In [ ]:
refined = {}
for name in SHORTLIST:
    refined[name] = wx.Experiment.from_bundle(task, bundles[name], verbose=VERBOSE)
    print(f"\n=== {name} ===")
    display(refined[name].select_features(n_trials=FEATURE_TRIALS, retune_trials=RETUNE_TRIALS))

print("Walk-forward folds:")
refined[SHORTLIST[0]].folds_table()

---
## 4. Select the champion (stage 5)

The rule is **highest mean macro F1 over the walk-forward folds**, fixed before the results were seen.

Macro F1 weights every hazard class equally, so the selection cannot be won by a model that only
predicts the common Low Risk days. Check the test confusion matrix in section 6 as well.

`typical_fold_std` shows how much a score moves from one year to the next. If the champion's
margin over the runner-up is smaller than that, the two are not clearly separated, and simplicity,
cost or interpretability can decide between them.

In [ ]:
refinement = wx.compare_refined(task, refined)
champion = wx.select_champion(refinement)
CHAMPION = champion["champion"]

wx.plot_cv_comparison(task, refinement, f"{task.name}-feature-selection")
plt.show()
print(f"Champion : {CHAMPION} (score {champion['score']:.4f})")
print(f"Runner-up: {champion['runner_up']} (margin {champion['margin']:.4f}, "
      f"typical fold spread {champion['typical_fold_std']:.4f})")
for message in champion["warnings"]:
    print(f"WARNING: {message}")
display(refinement)
refined[CHAMPION].cv_summary

---
## 5. Final fit (stage 6)

The champion's settings and features are refitted on **all development data** (2016–2024). Every
decision was made with cross-validation, so the most recent year can now be used for training.

In [ ]:
final_models = refined[CHAMPION].refit_final()
FINAL_MODELS = {wx.horizon_from_name(t): m for t, m in final_models.items()}
FEATURES = {wx.horizon_from_name(t): c for t, c in refined[CHAMPION].best_columns.items()}
{h: len(c) for h, c in FEATURES.items()}   # number of features used per horizon

---
## 6. Single test evaluation (stage 7)

The test split (2025) is loaded and scored **once**, for the champion and for the baselines (also
fitted on all development data). Nothing after this point changes any decision.

Expect the test score to be a little worse than the CV score: that is the selection optimism being
paid back. A large gap belongs in the limitations section of the report.

In [ ]:
X, y = wx.load_splits(task, ("train", "test"))
# Classification metrics are averaged over the classes seen in training.
LABELS = sorted(y["train"][task.target_columns[0]].unique()) if task.is_classification else None

test_rows = (wx.score_models(task, CHAMPION, FINAL_MODELS, FEATURES, X, y, "test", LABELS)
             + wx.score_baselines(task, X, y, "test", LABELS))
test_scores = pd.DataFrame(test_rows).drop(columns=["split"]).set_index(["target", "experiment"])

optimism = pd.DataFrame({
    "walk_forward_cv": refined[CHAMPION].cv_summary["cv_mean"],
    "test": test_scores.xs(CHAMPION, level="experiment")[task.metric],
})
optimism["test_minus_cv"] = optimism["test"] - optimism["walk_forward_cv"]
print(f"Selection optimism for {CHAMPION} ({task.metric}):")
display(optimism)

wx.plot_test_predictions(task, FINAL_MODELS, FEATURES, X, y,
                         f"{CHAMPION}: test predictions", f"{task.name}-test")
plt.show()
test_scores

---
## 7. Promote to production (stage 8)

The champion is written to `models/weather_hazard/production/`, replacing whatever was there:

| File | Contents |
|:--|:--|
| `<champion>_h<horizon>.joblib` | The final models, fitted on all development data |
| `metadata.json` | The `/model-metadata` document: CV and test metrics, features and importance per horizon |
| `manifest.json` | Model files and **each model's own feature list** (the API selects these columns) |
| `drift_reference.json` | The training distribution of every feature used, for drift monitoring |
| `promotion.json` | The audit record: ranking, shortlist, feature selection, test and warnings |

> The FastAPI service loads from `production/` (copied with `make artefacts`) and compares recently
> served feature rows with `drift_reference.json` at `/monitoring/drift`.

In [ ]:
record = wx.promotion_record(task, champion, ranking, shortlist_table, refinement,
                             test_scores, refined[CHAMPION])
destination = wx.promote_champion(task, refined[CHAMPION], test_scores, record)

print(f"Promoted {CHAMPION} -> {destination.relative_to(task.root)}")
sorted(path.name for path in destination.iterdir())

---
## 8. Conclusions

Points to carry into the report:

- **The ranking after tuning.** Which algorithms were shortlisted, and by what margin.
- **What feature selection changed.** Which families were dropped for which horizon, and whether
  the re-tune helped; this is the quantitative answer to "which features matter".
- **The champion, and how clear the win was.** Compare the margin over the runner-up with the
  typical fold spread.
- **Quote the test score, not the CV score,** and report the optimism between them.
- **Against the baselines on test.** Improvement over the naive forecasts is more informative than
  any absolute number.
- **Monitoring.** The production bundle carries a drift reference; the API reports when served
  data leaves the training range, which is the trigger for retraining.

> Rerun this notebook whenever an experiment notebook is rerun, so the promotion record never
> refers to results that have since changed.